In [ ]:
# Cargamos librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    classification_report
)

In [ ]:
df = pd.read_excel("Datos_CLV.xlsx", sheet_name="Histórico_Ventas")

In [ ]:
# Revisamos los nombres reales de las columnas (en Excel se ven cortados)
df.columns = df.columns.str.strip()
print(df.columns.tolist())

In [ ]:
df.head()

In [ ]:
# Nombres de las columnas tal como aparecen en la hoja Histórico_Ventas
col_precio = "precio_unitari_o"
col_unidades = "total_unidades_de_producto"
col_marca = "marca_"
print(col_precio, col_unidades, col_marca)

In [ ]:
# Convertimos la variable objetivo a categórica:
# conservamos TODAS las marcas que tengan al menos 'min_frec' registros
# y solo las marcas muy poco frecuentes se agrupan en 'OTRAS'
# (entre más bajo sea min_frec, más marcas salen por separado; debe ser >= 2 por el stratify)
min_frec = 20
df = df.dropna(subset=[col_marca]).copy()   # quitamos registros sin marca
df[col_marca] = df[col_marca].astype(str).str.strip()
frecuencias = df[col_marca].value_counts()
marcas_frecuentes = frecuencias[frecuencias >= min_frec].index
df['Marca_cat'] = df[col_marca].where(df[col_marca].isin(marcas_frecuentes), 'OTRAS')

print('Marcas distintas en total:', len(frecuencias))
print('Marcas que se conservan por separado:', len(marcas_frecuentes))

# Revisamos los valores de la variable objetivo
df['Marca_cat'].value_counts(dropna=False)

In [ ]:
# Eliminamos los registros que no tienen categoría objetivo
df = df.dropna(subset=['Marca_cat']).copy()

# Otras columnas que usaremos como predictoras
col_estado = 'estado_provenientw'
col_vendedor = 'vendedor_EWS'
col_fecha = 'fecha_documento_PROVENIENTE'

In [ ]:
# TRANSFORMACIÓN 1: variables numéricas
# Convertimos a número, rellenamos nulos con la mediana y quitamos valores negativos
for c in [col_precio, col_unidades]:
    df[c] = pd.to_numeric(df[c], errors='coerce')
    df[c] = df[c].fillna(df[c].median())
    df[c] = df[c].clip(lower=0)

# Recortamos los outliers extremos (percentil 1 y 99)
for c in [col_precio, col_unidades]:
    df[c] = df[c].clip(df[c].quantile(0.01), df[c].quantile(0.99))

# Precio y unidades suelen tener sesgo a la derecha: aplicamos logaritmo
df['log_precio'] = np.log1p(df[col_precio])
df['log_unidades'] = np.log1p(df[col_unidades])

In [ ]:
# TRANSFORMACIÓN 2: fecha -> día de la semana y día del mes
df[col_fecha] = pd.to_datetime(df[col_fecha], dayfirst=True, errors='coerce')
df[col_fecha] = df[col_fecha].fillna(df[col_fecha].mode()[0])
df['dia_semana'] = df[col_fecha].dt.dayofweek
df['dia_mes'] = df[col_fecha].dt.day

In [ ]:
# TRANSFORMACIÓN 3: variables categóricas (estado y vendedor) -> variables dummy
# Agrupamos las categorías poco frecuentes en 'OTROS' con el mismo umbral min_frec
for c, nueva in [(col_estado, 'Estado_cat'), (col_vendedor, 'Vendedor_cat')]:
    df[c] = df[c].fillna('SIN DATO').astype(str).str.strip()
    frec = df[c].value_counts()
    df[nueva] = df[c].where(df[c].isin(frec[frec >= min_frec].index), 'OTROS')

dummies = pd.get_dummies(df[['Estado_cat', 'Vendedor_cat']], drop_first=True, dtype=int)
df = pd.concat([df, dummies], axis=1)
print('Variables dummy creadas:', dummies.shape[1])

In [ ]:
# Declaramos las variables dependientes e independientes
columnas_modelo = ['log_precio', 'log_unidades', 'dia_semana', 'dia_mes'] + list(dummies.columns)
Vars_Indep = df[columnas_modelo]
Var_Dep = df['Marca_cat']
print(Vars_Indep.shape)

In [ ]:
#Redefinimos las variables
X = Vars_Indep
y = Var_Dep

In [ ]:
# Dividimos el conjunto de datos en entrenamiento y prueba
# stratify conserva, en lo posible, la proporción de cada clase
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
#Se escalan todos los datos
escalar = StandardScaler()

In [ ]:
# Ajustamos el escalador con los datos de entrenamiento
# y aplicamos la misma transformación a los datos de prueba
X_train = escalar.fit_transform(X_train)
X_test = escalar.transform(X_test)

In [ ]:
# Definimos el algoritmo de regresión logística multinomial
# Con más de dos clases, scikit-learn ajusta el modelo multinomial
algoritmo = LogisticRegression(
    solver='lbfgs',
    max_iter=1000
)

In [ ]:
#Entrenamos el modelo
algoritmo.fit(X_train, y_train)

In [ ]:
#Realizamos una predicción
y_pred = algoritmo.predict(X_test)
y_pred

In [ ]:
# Consultamos las categorías que predice el modelo
print('Clases del modelo:')
print(algoritmo.classes_)

In [ ]:
# Verificamos la matriz de confusión
matriz = confusion_matrix(y_test, y_pred, labels=algoritmo.classes_)
print('Matriz de confusión:')
print(matriz)

In [ ]:
# Visualizamos la matriz de confusión
disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=algoritmo.classes_
)
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(cmap='Blues', values_format='d', ax=ax, xticks_rotation=45)
plt.title('Matriz de confusión - Regresión logística multinomial')
plt.show()

In [ ]:
# Calculamos la precisión del modelo
# average='weighted' considera el número de observaciones de cada clase
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
print('Precisión ponderada del modelo:')
print(precision)

In [ ]:
# Calculamos la exactitud del modelo
exactitud = accuracy_score(y_test, y_pred)
print('Exactitud del modelo:')
print(exactitud)

In [ ]:
# Calculamos la sensibilidad (recall) ponderada
sensibilidad = recall_score(y_test, y_pred, average='weighted', zero_division=0)
print('Sensibilidad ponderada del modelo:')
print(sensibilidad)

In [ ]:
# Calculamos el puntaje F1 ponderado
puntajef1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print('Puntaje F1 ponderado del modelo:')
print(puntajef1)

In [ ]:
# Reporte de clasificación por cada categoría
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Probabilidades estimadas para cada categoría
probabilidades = algoritmo.predict_proba(X_test)
df_probabilidades = pd.DataFrame(
    probabilidades,
    columns=[f'Probabilidad_{clase}' for clase in algoritmo.classes_]
)
df_probabilidades.head()

In [ ]:
# Comparamos la categoría real con la categoría predicha
comparacion = pd.DataFrame({
    'Clase real': y_test.reset_index(drop=True),
    'Clase predicha': pd.Series(y_pred)
})
comparacion.head(10)